## ⚠️ CELL 0 — Run This First: PyTorch P100 Compatibility Fix

**The Kaggle default PyTorch 2.10.0+cu128 does NOT support the Tesla P100 (CUDA sm_60).**
Run this cell first to install a compatible PyTorch version, then **restart the kernel** before continuing.

> After this cell finishes, go to **Runtime → Restart kernel**, then run all remaining cells.
  You only need to do this once per session.

In [ ]:
# ── PyTorch P100 Compatibility Fixer ─────────────────────────────────────────
# The Kaggle default PyTorch 2.10+cu128 requires sm_70+ (Volta/Ampere).
# Tesla P100 is sm_60 (Pascal). Fix: install PyTorch 2.0.1+cu118 which supports sm_60.
import subprocess, sys

def check_p100_and_fix():
    try:
        import torch
        if not torch.cuda.is_available():
            print('[INFO] No GPU detected — skipping compatibility fix.')
            return

        cap = torch.cuda.get_device_capability(0)
        gpu = torch.cuda.get_device_name(0)
        print(f'[INFO] GPU: {gpu}  |  CUDA capability: sm_{cap[0]}{cap[1]}')

        if cap[0] < 7:  # sm_60 (P100) or lower
            print('[WARNING] This GPU needs an older PyTorch. Installing PyTorch 2.0.1+cu118...')
            subprocess.check_call([
                sys.executable, '-m', 'pip', 'install', '-q',
                'torch==2.0.1+cu118',
                'torchvision==0.15.2+cu118',
                '--extra-index-url', 'https://download.pytorch.org/whl/cu118',
                '--force-reinstall'
            ])
            print('\n[DONE] PyTorch 2.0.1+cu118 installed.')
            print('[ACTION REQUIRED] *** RESTART KERNEL NOW, then run all cells below ***')
        else:
            print(f'[OK] GPU sm_{cap[0]}{cap[1]} is compatible with the current PyTorch. No fix needed.')
    except Exception as e:
        print(f'[ERROR] Compatibility check failed: {e}')

check_p100_and_fix()

# 🚀 Version 3.0 — Sentinel-2 Road Extraction via Multi-Teacher Knowledge Distillation
### 📚 Sirko et al. (2024) *High-Resolution Building and Road Detection from Sentinel-2* (arXiv:2310.11622)

**Kaggle P100 / T4 optimised** — adjusted to parse nested `Sentinal-2` dataset layout.

---
### 🎯 Highlights
| Feature | Detail |
|---------|--------|
| Dual Teachers | Baseline DeepLabV3+ + FocusMIM DeepLabV3+ (both frozen) |
| Student | DeepLabV3+ ResNet-34 fresh init |
| Loss | α·(BCE+Dice vs GT) + β·SoftBCE vs teacher ensemble |
| Schedule | CosineAnnealingLR 30 epochs |
| AMP | `torch.cuda.amp` (P100-compatible) |
| Batch size | 8 images × 512×512 |
| Mask fix | White-pixel road check R>200 & G>200 & B>200 |

In [ ]:
# Install required packages (silent, Kaggle/Colab compatible)
!pip install -q segmentation-models-pytorch albumentations

import os, time, random, glob
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'PyTorch: {torch.__version__}')

## ⚙️ 1b. AMP Helper — Kaggle P100 Compatible

Kaggle P100 uses PyTorch ~1.13–2.x. We use `torch.cuda.amp` (stable across all versions).

In [ ]:
USE_AMP = device.type == 'cuda'
autocast = torch.cuda.amp.autocast
scaler_global = torch.cuda.amp.GradScaler(enabled=USE_AMP)
print(f'AMP enabled: {USE_AMP}')

## 👩‍🏫 2. Loading Dual Teacher Models

> **Upload checkpoints** as a Kaggle dataset titled `road-models`.  
> Expected filenames: `best_deeplabv3plus.pth` and `best_deeplabv3plus_focusmim.pth`.

| Teacher | Trained Val IoU |
|---------|----------------|
| Baseline DeepLabV3+ | 0.6637 |
| FocusMIM DeepLabV3+ | 0.6479 |

In [ ]:
ENCODER         = 'resnet34'
ENCODER_WEIGHTS = 'imagenet'
NUM_CLASSES     = 1
IMG_SIZE        = 512

def build_deeplabv3plus():
    return smp.DeepLabV3Plus(
        encoder_name    = ENCODER,
        encoder_weights = ENCODER_WEIGHTS,
        in_channels     = 3,
        classes         = NUM_CLASSES,
    )

def find_ckpt(candidates):
    return next((p for p in candidates if os.path.exists(p)), None)

t1_path = find_ckpt([
    '/kaggle/input/road-models/best_deeplabv3plus.pth',
    'best_deeplabv3plus.pth',
])
t2_path = find_ckpt([
    '/kaggle/input/road-models/best_deeplabv3plus_focusmim.pth',
    'best_deeplabv3plus_focusmim.pth',
])

teacher_baseline = build_deeplabv3plus().to(device)
teacher_focusmim = build_deeplabv3plus().to(device)

if t1_path:
    teacher_baseline.load_state_dict(torch.load(t1_path, map_location=device))
    print(f'  Baseline Teacher loaded : {t1_path}')
else:
    print('  [WARN] Baseline Teacher not found — using ImageNet init')

if t2_path:
    teacher_focusmim.load_state_dict(torch.load(t2_path, map_location=device))
    print(f'  FocusMIM Teacher loaded : {t2_path}')
else:
    print('  [WARN] FocusMIM Teacher not found — using ImageNet init')

for p in teacher_baseline.parameters(): p.requires_grad = False
for p in teacher_focusmim.parameters(): p.requires_grad = False
teacher_baseline.eval()
teacher_focusmim.eval()
print('[OK] Teachers frozen and in eval mode.')

## 📁 3. Dataset Path Resolution (Auto-detecting nested Sentinal-2 structure)

Automatically detects paths for images and masks inside `/kaggle/input/sentinal-2/`:
- Images folder: `Sentinal-2/images_enhanced_png/images_enhanced_png/`
- Masks folder: `Sentinal-2/masks_png/masks_png/`

In [ ]:
def find_sentinel2_dirs():
    possible_roots = [
        '/kaggle/input',
        '.',
        '..',
    ]
    
    # 1. Search for directory containing 'sentinal-2' (case-insensitive)
    for root in possible_roots:
        if not os.path.exists(root): continue
        for path in glob.glob(os.path.join(root, '*')):
            basename = os.path.basename(path).lower()
            if 'sentinal-2' in basename or 'sentinel' in basename or 'sentinal' in basename:
                # Find nested folders inside the matched root
                nested_images = sorted(glob.glob(os.path.join(path, '**/images_enhanced_png'), recursive=True))
                nested_masks  = sorted(glob.glob(os.path.join(path, '**/masks_png'), recursive=True))
                
                if nested_images and nested_masks:
                    # Pick the deepest folder containing target assets
                    img_dir  = nested_images[-1]
                    mask_dir = nested_masks[-1]
                    print(f'[INFO] Auto-detected Image Dir : {img_dir}')
                    print(f'[INFO] Auto-detected Mask Dir  : {mask_dir}')
                    return img_dir, mask_dir
                    
    # 2. Hardcoded fallback path
    fallback_img  = '/kaggle/input/sentinal-2/images_enhanced_png/images_enhanced_png'
    fallback_mask = '/kaggle/input/sentinal-2/masks_png/masks_png'
    if os.path.exists(fallback_img) and os.path.exists(fallback_mask):
        print(f'[INFO] Using Fallback Image Dir : {fallback_img}')
        print(f'[INFO] Using Fallback Mask Dir  : {fallback_mask}')
        return fallback_img, fallback_mask
        
    raise FileNotFoundError("Sentinel-2 dataset structure not found. Please upload dataset containing images_enhanced_png and masks_png.")

IMG_DIR, MASK_DIR = find_sentinel2_dirs()

def get_paired_files(img_d, mask_d):
    # Get all images (png/jpg/jpeg)
    all_imgs = sorted(
        glob.glob(os.path.join(img_d, '*.png')) +
        glob.glob(os.path.join(img_d, '*.jpg')) +
        glob.glob(os.path.join(img_d, '*.jpeg'))
    )
    pairs = []
    for img_path in all_imgs:
        name = os.path.basename(img_path)
        # Candidate mask paths matching the basename
        candidates = [
            os.path.join(mask_d, name),
            os.path.join(mask_d, name.replace('.jpg', '.png')),
            os.path.join(mask_d, name.replace('.jpeg', '.png')),
        ]
        for cand in candidates:
            if os.path.exists(cand):
                pairs.append((img_path, cand))
                break
    print(f'[INFO] Paired {len(pairs)} image-mask samples.')
    return pairs

all_pairs = get_paired_files(IMG_DIR, MASK_DIR)

train_val, test_pairs  = train_test_split(all_pairs, test_size=0.10, random_state=SEED)
train_pairs, val_pairs = train_test_split(train_val,  test_size=0.111, random_state=SEED)
print(f'Train={len(train_pairs)} | Val={len(val_pairs)} | Test={len(test_pairs)}')

## 🧪 4. Sentinel-2 Enhancement & Mask Binarization

- Enhancement applied as **uint8 before Albumentations** normalization
- Road = white pixel: `R>200 AND G>200 AND B>200`

In [ ]:
def clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=1.2, tileGridSize=(8,8)).apply(l)
    return cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2RGB)

def bilateral(img):
    return cv2.bilateralFilter(img, 5, 25, 25)

def sharpen(img):
    k = np.array([[0,-0.2,0],[-0.2,1.8,-0.2],[0,-0.2,0]], np.float32)
    return np.clip(cv2.filter2D(img,-1,k),0,255).astype(np.uint8)

def enhance(img_rgb):
    return sharpen(bilateral(clahe(img_rgb)))

def binarize_mask(mask_rgb):
    return (
        (mask_rgb[:,:,0] > 200) &
        (mask_rgb[:,:,1] > 200) &
        (mask_rgb[:,:,2] > 200)
    ).astype(np.float32)

print('[OK] Preprocessing functions ready.')

## 🗃️ 5. Dataset & DataLoaders

> Teacher soft targets generated **inside training loop on GPU** — NOT in `__getitem__`.
  This prevents the `CUDA in forked subprocess` crash with `num_workers > 0` on Kaggle.

In [ ]:
train_tfm = A.Compose([ 
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
    A.OneOf([
        A.RandomBrightnessContrast(0.2, 0.2, p=1.0),
        A.HueSaturationValue(10, 20, 10, p=1.0),
    ], p=0.4),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

val_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

class Sentinel2Dataset(Dataset):
    def __init__(self, pairs, tfm=None):
        self.pairs = pairs
        self.tfm   = tfm

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        sp, mp = self.pairs[idx]

        img = cv2.cvtColor(cv2.imread(sp),  cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
        img = enhance(img)  # uint8 — BEFORE Albumentations normalize

        msk = cv2.cvtColor(cv2.imread(mp),  cv2.COLOR_BGR2RGB)
        msk = cv2.resize(msk, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        gt  = binarize_mask(msk)  # float32 H x W

        if self.tfm:
            out   = self.tfm(image=img, mask=gt)
            img_t = out['image']              # float32  C x H x W
            gt_t  = out['mask'].unsqueeze(0)  # float32  1 x H x W
        else:
            img_t = torch.from_numpy(img.transpose(2,0,1)).float() / 255.0
            gt_t  = torch.from_numpy(gt).unsqueeze(0)

        return img_t, gt_t

BATCH     = 8
NW        = 4 if device.type == 'cuda' else 0
PERSIST   = NW > 0

train_ds = Sentinel2Dataset(train_pairs, tfm=train_tfm)
val_ds   = Sentinel2Dataset(val_pairs,   tfm=val_tfm)
test_ds  = Sentinel2Dataset(test_pairs,  tfm=val_tfm)

train_dl = DataLoader(train_ds, BATCH, shuffle=True,  num_workers=NW, pin_memory=True, drop_last=True,  persistent_workers=PERSIST)
val_dl   = DataLoader(val_ds,   BATCH, shuffle=False, num_workers=NW, pin_memory=True, drop_last=False, persistent_workers=PERSIST)
test_dl  = DataLoader(test_ds,  BATCH, shuffle=False, num_workers=NW, pin_memory=True, drop_last=False, persistent_workers=PERSIST)

print(f'Train={len(train_dl)} | Val={len(val_dl)} | Test={len(test_dl)} batches')

## 📐 6. Hybrid Multi-Teacher Distillation Loss

```
L = α × [0.5·BCE(logits, hard_GT) + 0.5·Dice(logits, hard_GT)]
  + β × SoftBCE(logits, teacher_ensemble_prob)
```

teacher_ensemble = 0.5·σ(T_baseline) + 0.5·σ(T_focusmim)

**SoftBCE** (not MSE) gives the correct gradient signal for soft probability targets.

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0): super().__init__(); self.s = smooth
    def forward(self, logits, tgt):
        p = torch.sigmoid(logits).view(-1); t = tgt.view(-1)
        return 1.0 - (2.0*(p*t).sum()+self.s)/(p.sum()+t.sum()+self.s)

class DistillLoss(nn.Module):
    def __init__(self, alpha=0.6, beta=0.4):
        super().__init__()
        self.bce  = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.a    = alpha
        self.b    = beta

    def forward(self, logits, gt, soft):
        sup = 0.5*self.bce(logits, gt) + 0.5*self.dice(logits, gt)
        kd  = F.binary_cross_entropy_with_logits(logits, soft)
        return self.a*sup + self.b*kd

@torch.no_grad()
def metrics(probs, gt, thr=0.5, eps=1e-6):
    pred = (probs>thr).float().view(-1); g = gt.view(-1)
    tp = (pred*g).sum().item(); fp = (pred*(1-g)).sum().item(); fn = ((1-pred)*g).sum().item()
    return dict(iou=(tp+eps)/(tp+fp+fn+eps), dice=(2*tp+eps)/(2*tp+fp+fn+eps))

print('[OK] Loss and metrics defined.')

## 🏗️ 7. Student Model — DeepLabV3+ ResNet-34

In [ ]:
student = build_deeplabv3plus().to(device)

with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
    out   = student(dummy)
    print(f'Output : {out.shape}')
    print(f'Params : {sum(p.numel() for p in student.parameters()):,}')

## 🔄 8. Training Engine

- **Teacher inference inside training loop** (GPU, safe for multiprocessing)
- **Gradient clipping** at 1.0
- **torch.cuda.amp** (P100-safe)

In [ ]:
@torch.no_grad()
def teacher_soft(imgs, t1, t2):
    """Frozen teacher ensemble → soft probability target."""
    return (0.5*torch.sigmoid(t1(imgs)) + 0.5*torch.sigmoid(t2(imgs))).detach()

def train_epoch(model, dl, criterion, opt, sc, t1, t2):
    model.train()
    tl = ti = td = n = 0
    pbar = tqdm(dl, desc='Train', leave=False)
    for imgs, gt in pbar:
        imgs, gt = imgs.to(device, non_blocking=True), gt.to(device, non_blocking=True)
        bs = imgs.size(0)
        soft = teacher_soft(imgs, t1, t2)

        opt.zero_grad()
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, gt, soft)

        sc.scale(loss).backward()
        sc.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        sc.step(opt); sc.update()

        with torch.no_grad():
            probs = torch.sigmoid(logits).detach()
            m = metrics(probs, gt)

        tl += loss.item()*bs; ti += m['iou']*bs; td += m['dice']*bs; n += bs
        pbar.set_postfix(loss=f"{loss.item():.4f}", iou=f"{m['iou']:.4f}")

    return dict(loss=tl/n, iou=ti/n, dice=td/n)

@torch.no_grad()
def val_epoch(model, dl, criterion, t1, t2):
    model.eval()
    tl = ti = td = n = 0
    pbar = tqdm(dl, desc='Val  ', leave=False)
    for imgs, gt in pbar:
        imgs, gt = imgs.to(device, non_blocking=True), gt.to(device, non_blocking=True)
        bs = imgs.size(0)
        soft = teacher_soft(imgs, t1, t2)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, gt, soft)
        probs = torch.sigmoid(logits)
        m = metrics(probs, gt)
        tl += loss.item()*bs; ti += m['iou']*bs; td += m['dice']*bs; n += bs
        pbar.set_postfix(loss=f"{loss.item():.4f}", iou=f"{m['iou']:.4f}")

    return dict(loss=tl/n, iou=ti/n, dice=td/n)

print('[OK] Training engine ready.')

## ⚡ 9. Run Distillation Training (30 Epochs)

| Hyperparameter | Value |
|----------------|-------|
| Optimizer | AdamW lr=3e-4, wd=1e-4 |
| Scheduler | CosineAnnealingLR T=30 η_min=1e-6 |
| Loss weights | α=0.6 (GT) β=0.4 (KD) |
| Gradient clip | 1.0 |

In [ ]:
EPOCHS    = 30
SAVE_PATH = 'best_sentinel2_student_model.pth'

criterion  = DistillLoss(alpha=0.6, beta=0.4).to(device)
optimizer  = torch.optim.AdamW(student.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler     = torch.cuda.amp.GradScaler(enabled=USE_AMP)

history      = {k:[] for k in ('train_loss','val_loss','train_iou','val_iou','val_dice')}
best_val_iou = 0.0
t0 = time.time()

print(f'Epochs={EPOCHS} | Batch={BATCH} | Workers={NW} | AMP={USE_AMP}')
print('='*78)

for epoch in range(1, EPOCHS+1):
    tr = train_epoch(student, train_dl, criterion, optimizer, scaler, teacher_baseline, teacher_focusmim)
    va = val_epoch(  student, val_dl,   criterion,             teacher_baseline, teacher_focusmim)
    scheduler.step()

    for k, v in [('train_loss',tr['loss']),('val_loss',va['loss']),
                 ('train_iou', tr['iou']), ('val_iou', va['iou']),
                 ('val_dice',  va['dice'])]:
        history[k].append(v)

    tag = ''
    if va['iou'] > best_val_iou:
        best_val_iou = va['iou']
        torch.save(student.state_dict(), SAVE_PATH)
        tag = '  ⭐ saved'

    print(f"Ep [{epoch:02d}/{EPOCHS}] "
          f"Train loss={tr['loss']:.4f} iou={tr['iou']:.4f} | "
          f"Val  loss={va['loss']:.4f} iou={va['iou']:.4f} dice={va['dice']:.4f}{tag}")

print('='*78)
print(f'Done in {(time.time()-t0)/60:.1f} min | Best Val IoU: {best_val_iou:.4f}')

## 📊 10. Convergence Curves

In [ ]:
xs = range(1, EPOCHS+1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(xs, history['train_loss'], 'o-', c='#e74c3c', label='Train')
axes[0].plot(xs, history['val_loss'],   's-', c='#3498db', label='Val')
axes[0].set_title('Loss', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(xs, history['train_iou'], 'o-', c='#f39c12', label='Train')
axes[1].plot(xs, history['val_iou'],   's-', c='#2ecc71', label='Val')
axes[1].set_title('IoU', fontweight='bold'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(xs, history['val_dice'], 'd-', c='#9b59b6', label='Val Dice')
axes[2].set_title('Dice', fontweight='bold'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('V3.0 Sentinel-2 Distillation Convergence', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('v3_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔬 11. 5-Column Prediction Grid (10 Test Samples)

Columns: **Input** | **GT Mask** | **Baseline Teacher** | **FocusMIM Teacher** | **Distilled Student (overlay + IoU)**

In [ ]:
def predict_binary(model, inp_tensor):
    with torch.no_grad():
        return (torch.sigmoid(model(inp_tensor)) > 0.5).float().squeeze().cpu().numpy()

def visualize_grid(pairs, n=10):
    n = min(n, len(pairs))
    if os.path.exists(SAVE_PATH):
        student.load_state_dict(torch.load(SAVE_PATH, map_location=device))
    student.eval(); teacher_baseline.eval(); teacher_focusmim.eval()

    fig, axes = plt.subplots(n, 5, figsize=(25, 4.8*n))
    fig.suptitle('V3.0 Multi-Teacher vs Student Predictions', fontsize=17, fontweight='bold')

    for i, (sp, mp) in enumerate(pairs[:n]):
        raw = cv2.cvtColor(cv2.imread(sp), cv2.COLOR_BGR2RGB)
        raw = cv2.resize(raw, (IMG_SIZE, IMG_SIZE))
        prep = enhance(raw)

        msk_rgb = cv2.cvtColor(cv2.imread(mp), cv2.COLOR_BGR2RGB)
        msk_rgb = cv2.resize(msk_rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        gt = binarize_mask(msk_rgb)

        aug = val_tfm(image=prep, mask=gt)
        inp = aug['image'].unsqueeze(0).to(device)

        tb = predict_binary(teacher_baseline, inp)
        tf = predict_binary(teacher_focusmim, inp)
        st = predict_binary(student,          inp)

        st_iou = (np.logical_and(st, gt).sum() + 1e-6) / (np.logical_or(st, gt).sum() + 1e-6)

        overlay = raw.copy()
        overlay[st == 1.0] = [0, 230, 100]
        blended = cv2.addWeighted(raw, 0.65, overlay, 0.35, 0)

        cols  = [prep,  gt,     tb,                  tf,                   blended]
        names = ['Input','GT', 'Baseline Teacher', 'FocusMIM Teacher', f'Student IoU={st_iou:.4f}']
        cmaps = [None, 'gray', 'gray', 'gray', None]

        for j, (im, nm, cm) in enumerate(zip(cols, names, cmaps)):
            axes[i,j].imshow(im, cmap=cm)
            axes[i,j].set_title(f'#{i+1} {nm}', fontsize=9, fontweight='bold')
            axes[i,j].axis('off')

    plt.tight_layout()
    plt.savefig('v3_prediction_grid.png', dpi=120, bbox_inches='tight')
    plt.show()

visualize_grid(test_pairs, n=10)

## 📋 12. Final Test-Set Evaluation & Comparison Table

In [ ]:
if os.path.exists(SAVE_PATH):
    student.load_state_dict(torch.load(SAVE_PATH, map_location=device))

student.eval()
ious, dices = [], []
with torch.no_grad():
    for imgs, gt in tqdm(test_dl, desc='Test'):
        imgs, gt = imgs.to(device), gt.to(device)
        p = torch.sigmoid(student(imgs))
        m = metrics(p, gt)
        ious.append(m['iou']); dices.append(m['dice'])

mean_iou  = float(np.mean(ious))
mean_dice = float(np.mean(dices))

print('\n' + '='*55)
print('  V3.0 FINAL TEST RESULTS')
print('='*55)
print(f'  IoU  : {mean_iou:.4f}')
print(f'  Dice : {mean_dice:.4f}')
print('='*55)

df = pd.DataFrame({
    'Model':    ['Baseline DeepLabV3+', 'FocusMIM DeepLabV3+', 'V3.0 Distilled Student'],
    'Val IoU':  [0.6637, 0.6479, round(mean_iou,  4)],
    'Val Dice': [0.7970, 0.7853, round(mean_dice, 4)],
    'Dataset':  ['DeepGlobe','DeepGlobe','Sentinel-2'],
})
print()
print(df.to_string(index=False))

## 💾 13. Download `best_sentinel2_student_model.pth`

In [ ]:
from IPython.display import FileLink, display

if os.path.exists(SAVE_PATH):
    mb = os.path.getsize(SAVE_PATH)/1024/1024
    print(f'Checkpoint: {SAVE_PATH} ({mb:.1f} MB)')
    display(FileLink(SAVE_PATH))
    try:
        from google.colab import files
        files.download(SAVE_PATH)
    except ImportError:
        print('[Kaggle] Use the Output tab to download the checkpoint.')
    print('\n V3.0 Sentinel-2 Teacher -> Student distillation complete!')
else:
    print(f'Not found: {SAVE_PATH}')